In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    cohen_kappa_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

from xgboost import XGBClassifier
import warnings
warnings.filterwarnings("ignore")

In [3]:
PATH = "/content/drive/MyDrive/Datasets/wbcatt"
TARGET = "label"
SEED = 42

In [4]:
train_df = pd.read_csv(f"{PATH}/pbc_attr_v1_train.csv")
test_df = pd.read_csv(f"{PATH}/pbc_attr_v1_test.csv")
val_df = pd.read_csv(f"{PATH}/pbc_attr_v1_val.csv")

display(train_df.head())

,label,cell_size,cell_shape,nucleus_shape,nuclear_cytoplasmic_ratio,chromatin_density,cytoplasm_vacuole,cytoplasm_texture,cytoplasm_colour,granule_type,granule_colour,granularity
0,Neutrophil,big,round,unsegmented-band,low,densely,no,clear,light blue,small,pink,yes
1,Neutrophil,big,irregular,unsegmented-band,low,densely,no,clear,light blue,small,pink,yes
2,Neutrophil,big,irregular,unsegmented-band,low,densely,no,clear,light blue,small,pink,yes
3,Neutrophil,big,irregular,unsegmented-band,low,densely,no,clear,light blue,small,pink,yes
4,Neutrophil,small,round,unsegmented-round,low,densely,no,clear,light blue,small,pink,yes


In [5]:
train_df.shape

(6169, 12)

In [6]:
class_counts = train_df[TARGET].value_counts().sort_index()

display(class_counts.to_frame())

,count
label,
Basophil,744
Eosinophil,1876
Lymphocyte,736
Monocyte,829
Neutrophil,1984


In [7]:
train_df.isnull().sum()

,0
label,0
cell_size,0
cell_shape,0
nucleus_shape,0
nuclear_cytoplasmic_ratio,0
chromatin_density,0
cytoplasm_vacuole,0
cytoplasm_texture,0
cytoplasm_colour,0
granule_type,0


**Label and Feature Column Divide**

In [8]:
X_train = train_df.drop(columns=[TARGET])
X_val = val_df.drop(columns=[TARGET])
X_test = test_df.drop(columns=[TARGET])

y_train = train_df[TARGET]
y_val = val_df[TARGET]
y_test = test_df[TARGET]

print('Train:', X_train.shape, '| labels:', y_train.shape)
print('Validation:', X_val.shape, '| labels:', y_val.shape)
print('Test:', X_test.shape, '| labels:', y_test.shape)

Train: (6169, 11) | labels: (6169,)
Validation: (1030, 11) | labels: (1030,)
Test: (3099, 11) | labels: (3099,)


**preprocessing**

In [ ]:
lb_encoder = LabelEncoder()
y_train = lb_encoder.fit_transform(y_train)

y_val = lb_encoder.transform(y_val)
y_test = lb_encoder.transform(y_test)

In [10]:
class_name = lb_encoder.classes_.tolist()
class_ids = np.arange(len(class_name))

label_mapping = pd.DataFrame({'class_id': class_ids, 'label': class_name})
display(label_mapping)

,class_id,label
0,0,Basophil
1,1,Eosinophil
2,2,Lymphocyte
3,3,Monocyte
4,4,Neutrophil


In [11]:
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

X_train_encoded = ohe.fit_transform(X_train)
X_val_encoded = ohe.transform(X_val)
X_test_encoded = ohe.transform(X_test)

print('Train:', X_train_encoded.shape)
print('Validation:', X_val_encoded.shape)
print('Test:', X_test_encoded.shape)

Train: (6169, 31)
Validation: (1030, 31)
Test: (3099, 31)
